# LASSO RIDGE Regularization

**Why is Regularization Required?**  
Regularization is a technique used in machine learning to reduce overfitting by penalizing large coefficients (weights) in a model.  

When a model learns too much from the training data, including noise and outliers, it performs very well on training data but poorly on unseen (test) data — this is called overfitting.  

Regularization helps prevent that by keeping the model simpler and more general.

Example:  
Without regularization:  
👉 Model learns noise → predicts perfectly on training data but fails on new data.  

With regularization:  
👉 Model ignores noise → performs slightly worse on training data but better on unseen data.

# 1. Import packages and observe dataset

In [8]:
# Import numerical libraries
import pandas as pd 
import numpy as np

# Import graphical plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# Import Linear Regression Machine Learning Libraries
from sklearn import preprocessing 
# from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

In [10]:
data = pd.read_csv(r"E:\WORK\FSDS\Daily Notes\ML Dataset\car-mpg.csv")
data.head()

,mpg,cyl,disp,hp,wt,acc,yr,origin,car_type,car_name
0,18.0,8,307.0,130,3504,12.0,70,1,0,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,0,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,0,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,0,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,0,ford torino


In [12]:
# Drop Car Name 
# Replace origin into 1,2,3... don't forget get_dummies
# Replace ? with nan
# Replace all nan with median

data = data.drop(['car_name'], axis = 1)
data['origin'] = data['origin'].replace({1: 'america' , 2: 'europe' , 3: 'asia'})
data = pd.get_dummies(data,columns = ['origin'],dtype=int)
data = data.replace('?', np.nan)
# data = data.apply(Lambda x: x.fillna(x.median()), axis = 0)

In [22]:
data = data.apply(pd.to_numeric, errors='coerce')

# Fill missing values with median only for numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns
data[numeric_cols] = data[numeric_cols].apply(lambda x: x.fillna(x.median()))

In [24]:
data.head()

,mpg,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
0,18.0,8,307.0,130.0,3504,12.0,70,0,1,0,0
1,15.0,8,350.0,165.0,3693,11.5,70,0,1,0,0
2,18.0,8,318.0,150.0,3436,11.0,70,0,1,0,0
3,16.0,8,304.0,150.0,3433,12.0,70,0,1,0,0
4,17.0,8,302.0,140.0,3449,10.5,70,0,1,0,0


We have to predict the mpg column given the features.

# 2. Model Building
Here we would like to scale the data as the columns are caried which would result in 1 column dominating the others.  
First we devide the data into independent (x) and dependent data (y) the we scale it.  

**Tip!:**  
*The reason we don't scale the entire data before & then devide it into train(x) & test(y) is because once we scale the data, the type(data_s) would be numpy.ndarray. It's impossible to devide this data when it's an array.*

Hence we devode type(data) pandas. DataFrame, then proceed to scaling it.

In [29]:
x = data.drop(['mpg'], axis = 1) # Independent Variable
y = data[['mpg']] # Dependent Variable

In [35]:
# Scaling the Data

x_s = preprocessing.scale(x)
x_s = pd.DataFrame(x_s, columns = x.columns) # Converting scaled data into DataFrame

y_s = preprocessing.scale(y)
y_s = pd.DataFrame(y_s, columns = y.columns) # Ideally train, test data should be in columns

In [37]:
x_s

,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
0,1.498191,1.090604,0.673118,0.630870,-1.295498,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
1,1.498191,1.503514,1.589958,0.854333,-1.477038,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
2,1.498191,1.196232,1.197027,0.550470,-1.658577,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
3,1.498191,1.061796,1.197027,0.546923,-1.295498,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
4,1.498191,1.042591,0.935072,0.565841,-1.840117,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
...,...,...,...,...,...,...,...,...,...,...
393,-0.856321,-0.513026,-0.479482,-0.213324,0.011586,1.621983,0.941412,0.773559,-0.497643,-0.461968
394,-0.856321,-0.925936,-1.370127,-0.993671,3.279296,1.621983,0.941412,-1.292726,-0.497643,2.164651
395,-0.856321,-0.561039,-0.531873,-0.798585,-1.440730,1.621983,0.941412,0.773559,-0.497643,-0.461968
396,-0.856321,-0.705077,-0.662850,-0.408411,1.100822,1.621983,0.941412,0.773559,-0.497643,-0.461968


In [39]:
y_s

,mpg
0,-0.706439
1,-1.090751
2,-0.706439
3,-0.962647
4,-0.834543
...,...
393,0.446497
394,2.624265
395,1.087017
396,0.574601


In [41]:
data.shape

(398, 11)

In [43]:
# Split into Train, Test Set
x_train, x_test, y_train, y_test = train_test_split(x_s, y_s, test_size = 0.20, random_state = 0)
x_train.shape

(318, 10)

# 2.a Simple Linear Model

In [46]:
# Fit simple linear model & find coefficients
regression_model = LinearRegression()
regression_model.fit(x_train, y_train)

for idx, col_name in enumerate(x_train.columns):
    print('The coefficient for {} is {}'.format(col_name, regression_model.coef_[0][idx]))

intercept = regression_model.intercept_[0]
print('The intercept is {}'.format(intercept))

The coefficient for cyl is 0.24297168181227313
The coefficient for disp is 0.2923555012814415
The coefficient for hp is -0.18342828140772874
The coefficient for wt is -0.6656888297642692
The coefficient for acc is 0.06522198269426219
The coefficient for yr is 0.3476276950232327
The coefficient for car_type is 0.3364881665039492
The coefficient for origin_america is 7464399952891.158
The coefficient for origin_asia is 6151924272936.219
The coefficient for origin_europe is 5872026888334.256
The intercept is -0.018188609724684175


## 2.b Regularized Lasso Regression

In [53]:
# alpha factor here is lambda (penalty term) which helps to reduce the magnitude of coeff

ridge_model = Ridge(alpha = 0.4)
ridge_model.fit(x_train, y_train)

print('Ridge model coef: {}'.format(ridge_model.coef_))
# As the data has 10 columns hence 10 coefficients appear here 

Ridge model coef: [[ 0.24242411  0.28008024 -0.18071842 -0.65711583  0.06353256  0.34721777
   0.32998816 -0.08077573  0.06989674  0.02945199]]


## 2.c Regularized Lasso Regression

In [56]:
# alpha factor here is lambda (penalty term) which helps to reduce the magnitude of coeff

lasso_model = Lasso(alpha = 0.1)
lasso_model.fit(x_train, y_train)

print('Lasso model coef: {}'.format(lasso_model.coef_))
# As the data has 10 columns hence 10 coefficients appear here 

Lasso model coef: [-0.         -0.         -0.07247557 -0.45867691  0.          0.2698134
  0.11341188 -0.04988145  0.          0.        ]
